# EDA: Age vs topics, popularity, and review sentiment

This notebook explores how reading age relates to Amazon category/topic, popularity signals (review counts, ratings, price), and sentiment extracted from review text (non-LLM).


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')


## Load master table

Prefers batch1 if present.


In [ ]:
def read_master():
    candidates = [
        Path('data/processed/master_batch1.parquet'),
        Path('data/processed/master_batch1.csv.gz'),
        Path('data/processed/master_stories.parquet'),
        Path('data/processed/master_stories.csv.gz'),
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError('No master file found in data/processed/')
    if path.suffix == '.parquet':
        df = pd.read_parquet(path)
    else:
        df = pd.read_csv(path, low_memory=False)
    return path, df

path, df = read_master()
print('Loaded:', path, df.shape)
df.head(2)


## Normalize key columns


In [ ]:
d = df.copy()
d['age_mid'] = pd.to_numeric(d.get('age_mid'), errors='coerce')
for c in ['Num Reviews','Average Rating','Price','yt_view_count','yt_like_count','yt_duration']:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors='coerce')

# Category leaf (last breadcrumb)
if 'Categories' in d.columns:
    d['category_leaf'] = d['Categories'].astype(str).str.split('>').str[-1].str.strip()
else:
    d['category_leaf'] = np.nan

d_age = d[d['age_mid'].notna()].copy()
print('Rows with age_mid:', len(d_age))


## Age distribution by category (top leaves)


In [ ]:
top = d_age['category_leaf'].value_counts().head(15)
top_names = top.index.tolist()
dd = d_age[d_age['category_leaf'].isin(top_names)].copy()

plt.figure(figsize=(10,5))
sns.boxplot(data=dd, x='age_mid', y='category_leaf', order=top_names)
plt.title('Age distribution by top Amazon leaf category')
plt.tight_layout()


## Popularity vs age (Num Reviews, view_count)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

if 'Num Reviews' in d_age.columns:
    x = d_age[d_age['Num Reviews'].notna()].copy()
    x['log_num_reviews'] = np.log10(x['Num Reviews'].clip(lower=1))
    sns.regplot(data=x, x='age_mid', y='log_num_reviews', scatter_kws={'s':8,'alpha':0.25}, ax=axes[0])
    axes[0].set_title('Age vs Num Reviews (log10)')
else:
    axes[0].axis('off')
    axes[0].text(0.1,0.5,'Num Reviews not present', transform=axes[0].transAxes)

if 'yt_view_count' in d_age.columns:
    x = d_age[d_age['yt_view_count'].notna()].copy()
    x['log_views'] = np.log10(x['yt_view_count'].clip(lower=1))
    sns.regplot(data=x, x='age_mid', y='log_views', scatter_kws={'s':8,'alpha':0.25}, ax=axes[1])
    axes[1].set_title('Age vs YouTube view_count (log10)')
else:
    axes[1].axis('off')
    axes[1].text(0.1,0.5,'yt_view_count not present', transform=axes[1].transAxes)

plt.tight_layout()


## Ratings and price vs age


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

if 'Average Rating' in d_age.columns:
    x = d_age[d_age['Average Rating'].notna()].copy()
    sns.regplot(data=x, x='age_mid', y='Average Rating', scatter_kws={'s':8,'alpha':0.25}, ax=axes[0])
    axes[0].set_ylim(0,5.05)
    axes[0].set_title('Age vs Average Rating')
else:
    axes[0].axis('off')

if 'Price' in d_age.columns:
    x = d_age[d_age['Price'].notna()].copy()
    sns.regplot(data=x, x='age_mid', y='Price', scatter_kws={'s':8,'alpha':0.25}, ax=axes[1])
    axes[1].set_title('Age vs Price')
else:
    axes[1].axis('off')

plt.tight_layout()


## Topic modeling (non-LLM) from Summary + Title

Uses TF-IDF + NMF to get coarse topics and see which age bins they correlate with.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

def make_doc(row):
    parts = []
    for c in ['Title','Summary']:
        if c in row and pd.notna(row[c]):
            parts.append(str(row[c]))
    return ' '.join(parts).strip()

docs = d_age.apply(make_doc, axis=1)
mask = docs.str.len() > 0
x = d_age[mask].copy()
docs = docs[mask]

bins = [0, 3, 6, 12]
labels = ['0-3','4-6','7-12']
x['age_bin'] = pd.cut(x['age_mid'], bins=bins, labels=labels, include_lowest=True)
x = x[x['age_bin'].notna()].copy()
docs = docs.loc[x.index]

vec = TfidfVectorizer(max_features=8000, stop_words='english', ngram_range=(1,2), min_df=5)
X = vec.fit_transform(docs)
n_topics = 8
nmf = NMF(n_components=n_topics, random_state=0, init='nndsvda', max_iter=400)
W = nmf.fit_transform(X)
H = nmf.components_
terms = np.array(vec.get_feature_names_out())

top_words = []
for k in range(n_topics):
    idx = np.argsort(H[k])[-12:][::-1]
    top_words.append(', '.join(terms[idx]))
topic_df = pd.DataFrame({'topic': range(n_topics), 'top_terms': top_words})
topic_df


In [ ]:
Wdf = pd.DataFrame(W, index=x.index, columns=[f'topic_{i}' for i in range(n_topics)])
tmp = pd.concat([x[['age_bin']], Wdf], axis=1)
means = tmp.groupby('age_bin')[[f'topic_{i}' for i in range(n_topics)]].mean()
plt.figure(figsize=(12,4))
sns.heatmap(means, cmap='viridis')
plt.title('Mean topic weight by age bin (TF-IDF + NMF)')
plt.tight_layout()


## Review sentiment (non-LLM)

Parses `Customer Reviews` JSON if present and computes VADER sentiment if available.


In [ ]:
def try_vader():
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        return SentimentIntensityAnalyzer()
    except Exception:
        return None

analyzer = try_vader()
print('VADER available:', analyzer is not None)


In [ ]:
def parse_reviews(cell):
    if cell is None:
        return []
    if isinstance(cell, float) and np.isnan(cell):
        return []
    if isinstance(cell, list):
        return cell
    s = str(cell).strip()
    if not s or s.lower() == 'nan':
        return []
    try:
        obj = json.loads(s)
        return obj if isinstance(obj, list) else []
    except Exception:
        return []

def review_text(r):
    if isinstance(r, dict):
        return str(r.get('text','') or '')
    return ''

def review_rating(r):
    if not isinstance(r, dict):
        return np.nan
    s = str(r.get('rating','') or '')
    try:
        return float(s.split()[0])
    except Exception:
        return np.nan

if 'Customer Reviews' not in d_age.columns:
    print('No Customer Reviews column in master; skipping sentiment section.')
else:
    rows = []
    for idx, row in d_age.iterrows():
        reviews = parse_reviews(row.get('Customer Reviews'))
        if not reviews:
            continue
        for r in reviews:
            txt = review_text(r)
            if not txt.strip():
                continue
            comp = np.nan
            if analyzer is not None:
                comp = analyzer.polarity_scores(txt).get('compound', np.nan)
            rows.append({
                'video_id': row.get('video_id'),
                'age_mid': row.get('age_mid'),
                'category_leaf': row.get('category_leaf'),
                'rating': review_rating(r),
                'sent_compound': comp,
                'text': txt,
            })
    rev = pd.DataFrame(rows)
    print('parsed reviews:', rev.shape)
    rev.head(3)


In [ ]:
if 'Customer Reviews' in d_age.columns and 'rev' in globals() and len(rev):
    agg = rev.groupby('video_id', as_index=False).agg({
        'sent_compound':'mean',
        'rating':'mean',
        'text':'count',
    }).rename(columns={'text':'n_reviews_parsed', 'rating':'mean_review_rating', 'sent_compound':'mean_review_sentiment'})
    merged = d_age.merge(agg, on='video_id', how='left')
    merged = merged[merged['mean_review_sentiment'].notna()].copy()

    plt.figure(figsize=(7,4))
    sns.regplot(data=merged, x='age_mid', y='mean_review_sentiment', scatter_kws={'s':10,'alpha':0.25})
    plt.title('Age vs mean review sentiment (VADER compound)')
    plt.tight_layout()

    plt.figure(figsize=(7,4))
    sns.regplot(data=merged, x='age_mid', y='mean_review_rating', scatter_kws={'s':10,'alpha':0.25})
    plt.ylim(0,5.05)
    plt.title('Age vs mean review rating (parsed)')
    plt.tight_layout()
else:
    print('No parsed review sentiment available (either no reviews or no VADER).')


## Category × popularity × age (small multiples)


In [ ]:
if 'Num Reviews' in d_age.columns:
    top = d_age['category_leaf'].value_counts().head(8).index.tolist()
    sub = d_age[d_age['category_leaf'].isin(top) & d_age['Num Reviews'].notna()].copy()
    sub['log_num_reviews'] = np.log10(sub['Num Reviews'].clip(lower=1))
    g = sns.lmplot(
        data=sub, x='age_mid', y='log_num_reviews', col='category_leaf', col_wrap=4,
        scatter_kws={'s':8,'alpha':0.2}, line_kws={'color':'black'}, height=3, aspect=1.1
    )
    g.fig.suptitle('Age vs Num Reviews by category (log10)', y=1.02)
    plt.tight_layout()
else:
    print('Num Reviews not present.')
